# 03 — Gold: dim_financial_segment

| Property | Value |
|----------|-------|
| **Gold Table** | `dim_financial_segment` |
| **Grain** | One row per GlobalFinancialSegmentId |
| **Source** | `ref.FinancialSegmentHierarchy` |
| **PK** | `GlobalFinancialSegmentId` (int) |
| **Rows** | 3,993 |

**Hierarchy**: Segment (8) → Business (57) → LOB (177) → ProductService (552) → Team (3,203)

**Cross-sell**: Segment level (CRB/HCB/IRR) is the main cross-sell axis.

In [ ]:
# ============================================================
# Cell 1: Setup & Config
# ============================================================
from pyspark.sql import functions as F

LAKEHOUSE = "The_Global_Loom"
TABLE = "dim_financial_segment"
SOURCE_TABLE = "ref.FinancialSegmentHierarchy"

print(f"Config: {SOURCE_TABLE} → {LAKEHOUSE}.{TABLE}")

In [ ]:
# ============================================================
# Cell 2: Read silver source
# ============================================================
df_src = spark.table(SOURCE_TABLE)

print(f"Source: {df_src.count():,} rows × {len(df_src.columns)} cols")
df_src.printSchema()

## Cell 3: Transform

- Keep hierarchy names and codes (drop Ids — use names for BI display)
- Drop: SecurityCode, all ETL dates
- Keep GlobalFinancialSegmentId as PK

In [ ]:
# ============================================================
# Cell 3: Transform
# ============================================================
df_clean = df_src.select(
    F.col("GlobalFinancialSegmentId").cast("int"),
    F.col("SegmentCode").cast("string"),
    F.col("SegmentName").cast("string"),
    F.col("BusinessCode").cast("string"),
    F.col("BusinessName").cast("string"),
    F.col("LOBCode").cast("string"),
    F.col("LOBName").cast("string"),
    F.col("ProductServiceCode").cast("string"),
    F.col("ProductServiceName").cast("string"),
    F.col("TeamCode").cast("string"),
    F.col("TeamName").cast("string"),
    F.col("IsDeleted").cast("boolean")
)

# Filter out deleted Unknown members (ID = -1 AND IsDeleted = True)
before_count = df_clean.count()
df_clean = df_clean.filter(~((F.col("GlobalFinancialSegmentId") == -1) & (F.col("IsDeleted") == True)))
after_count = df_clean.count()

if before_count > after_count:
    print(f"Filtered out {before_count - after_count} deleted Unknown member(s) (ID=-1, IsDeleted=True)")

print(f"After column select + filter: {after_count:,} rows × {len(df_clean.columns)} cols")

In [ ]:
# ============================================================
# Cell 4: Add Unknown member
# ============================================================

# Check if -1 still exists (meaning it had actual data and wasn't filtered)
existing_ids = df_clean.select("GlobalFinancialSegmentId").distinct().collect()
existing_id_set = {row.GlobalFinancialSegmentId for row in existing_ids}

unknown_id = -1 if -1 not in existing_id_set else -999

if unknown_id == -999:
    print(f"Using -999 for Unknown member (ID=-1 has actual data)")

unknown_row = spark.createDataFrame([(
    unknown_id, "Unknown", "Unknown", "Unknown", "Unknown",
    "Unknown", "Unknown", "Unknown", "Unknown",
    "Unknown", "Unknown", False
)], schema=df_clean.schema)

df_final = df_clean.unionByName(unknown_row)

print(f"Added Unknown member (ID={unknown_id}): {df_final.count():,} rows")

In [ ]:
# ============================================================
# Cell 5: Data quality checks
# ============================================================
total = df_final.count()
distinct_pks = df_final.select("GlobalFinancialSegmentId").distinct().count()
dupes = total - distinct_pks

print(f"DQ Checks")
print(f"   Total rows:     {total:,}")
print(f"   Distinct PKs:   {distinct_pks:,}")
print(f"   Duplicate PKs:  {dupes}")

if dupes > 0:
    print(f"\nERROR: Found {dupes} duplicate GlobalFinancialSegmentId values")
    print("\nShowing duplicate IDs and their counts:")
    df_final.groupBy("GlobalFinancialSegmentId") \
        .count() \
        .filter(F.col("count") > 1) \
        .orderBy(F.desc("count")) \
        .show(20, truncate=False)
    
    print("\nShowing full rows for duplicates:")
    duplicate_ids = [row.GlobalFinancialSegmentId for row in 
                     df_final.groupBy("GlobalFinancialSegmentId")
                     .count()
                     .filter(F.col("count") > 1)
                     .select("GlobalFinancialSegmentId")
                     .collect()]
    
    df_final.filter(F.col("GlobalFinancialSegmentId").isin(duplicate_ids)) \
        .orderBy("GlobalFinancialSegmentId") \
        .show(50, truncate=False)
    
    raise AssertionError(f"ERROR: Duplicates found! See details above.")

print("\nAll DQ checks passed")

In [ ]:
# ============================================================
# Cell 6: Write to gold lakehouse
# ============================================================
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE)

print(f"Written: {TABLE}")
print(f"   Rows: {spark.table(TABLE).count():,}")